### 1. Connect to our database and create a new database (if it does not exists already)

Note: If you are going to use a CodeSpace, the IP of the CodeSpace will differ from your local computer IP, thus you need to edit a rule to allow Codespace access to AWS. To check your codespace IP run:

```bash
curl -s https://checkip.amazonaws.com
```

Also, you may need to upgrade the sql connector:
```python
%pip install --upgrade mysql-connector-python
```


In [1]:
import mysql.connector
import time

conexion = mysql.connector.connect(
    host="database-3.cfy8okwu4c6q.eu-north-1.rds.amazonaws.com",
    port=3306,
    user="irampardo",
    password="CodeSpace0508",
    connection_timeout=10
)

cursor = conexion.cursor()

print("Connected:", conexion.is_connected())

start = time.time()
cursor.execute("SELECT 1")
print("Ping result:", cursor.fetchone())
print(f"Ping took {time.time() - start:.2f} seconds")

start = time.time()
cursor.execute("CREATE DATABASE IF NOT EXISTS panaderiasalvador")
print(f"Database command took {time.time() - start:.2f} seconds")

cursor.execute("USE panaderiasalvador")
print("Selected database:", conexion.database)


Connected: True
Ping result: (1,)
Ping took 0.03 seconds
Database command took 0.04 seconds
Selected database: panaderiasalvador


### 2. Create an empty database with each column and table name pre-defined

In [2]:
nombre_base_datos = "panaderiasalvador"
nombre_tabla_ventas = "raw_ventas"
nombre_tabla_calendario = "raw_calendario"
nombre_tabla_pedidos = "raw_pedidos"

cursor = conexion.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {nombre_base_datos}")

cursor.execute(f"USE {nombre_base_datos}")

In [3]:
cursor.execute(f"DROP TABLE IF EXISTS {nombre_tabla_ventas}")
cursor.execute(f"CREATE TABLE IF NOT EXISTS {nombre_tabla_ventas} ("
               "FAMILIA VARCHAR(255),"
               "Tipo VARCHAR(255),"
               "FechaVenta DATE,"
               "HoraVenta INT,"
               "Articulo VARCHAR(255),"
               "Cantidad FLOAT,"
               "Precio FLOAT,"
               "Importe FLOAT"
               ")")

In [4]:
cursor.execute(f"DROP TABLE IF EXISTS {nombre_tabla_calendario}")
cursor.execute(f"CREATE TABLE IF NOT EXISTS {nombre_tabla_calendario} ("
               "Fecha DATE,"
               "Festivo VARCHAR(255)"
               ")")

In [5]:
cursor.execute(f"DROP TABLE IF EXISTS {nombre_tabla_pedidos}")
cursor.execute(f"CREATE TABLE IF NOT EXISTS {nombre_tabla_pedidos} ("
               "Tipo VARCHAR(255),"
               "Fecha DATE,"
               "Articulo VARCHAR(255),"
               "Cantidad FLOAT,"
               "Precio FLOAT,"
               "Importe FLOAT"
               ")")

In [6]:
!pip install mysql.connector

import xlrd
import mysql.connector
import pandas as pd


In [7]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


### 3. Load .xlsx tables in each empty table created in step 2

In [8]:
# Load Excel file in CodeSpaces
nombre_archivo_ventas = "/workspaces/Datathon_CodeSpace/Data/ArticulosPanaderia.xlsx"

# Open Excel file
df = pd.read_excel(nombre_archivo_ventas)

In [9]:
# Prepare data for insertion
registros = []
for _, row in df.iterrows():
    familia = row['FAMILIA']
    tipo = row['Tipo']
    fecha_venta = row['FechaVenta'].strftime('%Y-%m-%d')
    hora_venta = row['HoraVenta']
    articulo = row['Articulo']
    cantidad = row['Cantidad']
    precio = row['Precio']
    importe = row['Importe']
    registros.append((familia, tipo, fecha_venta, hora_venta, articulo, cantidad, precio, importe))


In [10]:
# Load Excel file in CodeSpaces
nombre_archivo_ventas = "/workspaces/Datathon_CodeSpace/Data/ArticulosPanaderia.xlsx"

# Open the exel file using pandas
df = pd.read_excel(nombre_archivo_ventas)

# Prepare data for insertion into the database
registros = []
for _, row in df.iterrows():
    familia = row['FAMILIA']
    tipo = row['Tipo']
    fecha_venta = row['FechaVenta'].strftime('%Y-%m-%d')
    hora_venta = row['HoraVenta']
    articulo = row['Articulo']
    cantidad = row['Cantidad']
    precio = row['Precio']
    importe = row['Importe']
    registros.append((familia, tipo, fecha_venta, hora_venta, articulo, cantidad, precio, importe))

# Create the SQL query to insert the data into the table
consulta = f"INSERT INTO {nombre_tabla_ventas} (FAMILIA, Tipo, FechaVenta, HoraVenta, Articulo, Cantidad, Precio, Importe) " \
           "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"

# Execute the query with the list of records in batches
tamano_lote = 1000
for i in range(0, len(registros), tamano_lote):
    lote_registros = registros[i:i + tamano_lote]
    cursor = conexion.cursor()
    cursor.executemany(consulta, lote_registros)
    conexion.commit()

In [11]:
# Load Excel file in CodeSpaces
nombre_archivo_calendario = "/workspaces/Datathon_CodeSpace/Data/Calendario.xlsx"

# Open the Excel file using pandas
df = pd.read_excel(nombre_archivo_calendario)

# Prepare data for insertion into the database
registros = []
for _, row in df.iterrows():
    fecha = row['Fecha'].strftime('%Y-%m-%d')
    festivo = row['Festivo']
    registros.append((fecha,festivo))

# Create the SQL query to insert the data into the table
consulta = f"INSERT INTO {nombre_tabla_calendario} (Fecha, Festivo) " \
           "VALUES (%s, %s)"

# Execute the query with the list of records in batches
tamano_lote = 1000
for i in range(0, len(registros), tamano_lote):
    lote_registros = registros[i:i + tamano_lote]
    cursor = conexion.cursor()
    cursor.executemany(consulta, lote_registros)
    conexion.commit()

In [ ]:
# Load Excel file in CodeSpaces
nombre_archivo_pedidos = "/workspaces/Datathon_CodeSpace/Data/CantidadPedida.xlsx"

# Open the Excel file using pandas
df = pd.read_excel(nombre_archivo_pedidos)

# Prepare data for insertion into the database
registros = []
for _, row in df.iterrows():
    tipo = row['Tipo']
    fecha_venta = row['Fecha'].strftime('%Y-%m-%d')
    articulo = row['Articulo']
    cantidad = row['Cantidad']
    precio = row['Precio']
    importe = row['Importe']
    registros.append(( tipo, fecha_venta, articulo, cantidad, precio, importe))

# Create the SQL query to insert the data into the table
consulta = f"INSERT INTO {nombre_tabla_pedidos} (Tipo, Fecha, Articulo, Cantidad, Precio, Importe) " \
           "VALUES (%s, %s, %s, %s, %s, %s)"

# Execute the query with the list of records in batches
tamano_lote = 1000
for i in range(0, len(registros), tamano_lote):
    lote_registros = registros[i:i + tamano_lote]
    cursor = conexion.cursor()
    cursor.executemany(consulta, lote_registros)
    conexion.commit()

### 4. Launch queries to:
- Check database connection
- Order views
- Add Holiday days to calendar table

In [ ]:
consulta_proceso="""


/*Order products by family*/
drop table if exists data.articulos_top;
create table data.articulos_top as (

	select
	Articulo,
	FAMILIA,
	sum(importe) as importe_total,
	ROW_NUMBER () OVER(partition by FAMILIA order by sum(importe) desc) as orden
	from data.raw_ventas
	where FechaVenta >= '2021-05-01'
	group by 1,2

);


/*Create a general calendar from 1/1/2017 to 31/12/2023*/
drop table if exists data.calendario_dias;
create table data.calendario_dias as (

with recursive cte_calendario as (
	select date('2017-01-01') as calendar_date -- fecha de inicio
	union all
	select date_add(calendar_date, interval 1 day) as calendar_date from cte_calendario
	where date_add(calendar_date, interval 1 day) <= date('2023-12-31') -- condicion de fin, fecha final
)

select
calendar_date as fecha,
year(calendar_date) as fx_anno,
month(calendar_date) as fx_mes,
day(calendar_date) as fx_day,
date_format(calendar_date, '%Y%m') as fx_anno_mes,
date_format(calendar_Date,'%x-%v') as semana -- formato
from cte_calendario

);

/*Add holiday days to the calendar*/
drop table if exists data.calendario_completo;
create table data.calendario_completo as (
select
base.*,
festivos.festivo
from data.calendario_dias base
left join
	(select
	a.*,
	row_number() over(partition by a.fecha order by a.festivo) as orden
	from data.raw_calendario a
	) festivos
on base.fecha=festivos.fecha
and festivos.orden=1
);

/*Add sellings per day (remove hourly detail)*/

/*Add the product order and if it's in the study date range*/
drop table if exists data.ventas_diarias;
create table data.ventas_diarias as (
select
base.familia,
base.tipo,
base.fechaVenta,
calendario.festivo,
base.articulo,
sum(base.precio*base.cantidad)/sum(base.cantidad) as precio, /* Se calcula como el precio promedio del día, ya que hay cambios de precio a lo largo del día*/
articulos.orden as orden_articulo_familia,
case when base.fechaVenta >=date('2021-05-01') then 'S' else 'N' end as in_fecha_estudio,
sum(base.cantidad) as cantidad,
sum(base.importe) as importe

from data.raw_ventas base

inner join data.calendario_completo calendario
on base.FechaVenta=calendario.fecha

inner join data.articulos_top articulos
on base.familia=articulos.familia
and base.articulo=articulos.articulo


group by 1,2,3,4,5,7,8);


/*Create a final view filtering for this study*/
drop view if exists data.ventas_diarias_estudio_completo;
create view data.ventas_diarias_estudio_completo as
(select *
from data.ventas_diarias
where tipo='VENTA'
and in_fecha_estudio='S'
and orden_articulo_familia<=5);


/*Create an extra view with the filtering for the study without the days to predict (May 23rd)*/
drop view if exists data.ventas_diarias_estudio;
create view data.ventas_diarias_estudio as
(select *
from data.ventas_diarias
where tipo='VENTA'
and in_fecha_estudio='S'
and fechaventa < date('2023-05-01')
and orden_articulo_familia<=5);
"""

In [ ]:
# Separar las consultas en una lista
consultas_lista = consulta_proceso.split(";")

# Eliminar espacios en blanco y elementos vacíos
consultas_lista = [consulta.strip() for consulta in consultas_lista if consulta.strip()]

# Ejecutar las consultas
for consulta in consultas_lista:
    cursor.execute(consulta)

# Confirmar los cambios
conexion.commit()

OperationalError: 2055: Lost connection to MySQL server at 'database-1.cjyy8w6ya3fr.eu-north-1.rds.amazonaws.com:3306', system error: 60 Operation timed out

In [ ]:
# cursor.execute(f"grant select on {nombre_base_datos}.ventas_diarias to usuario1")
# cursor.execute(f"grant select on {nombre_base_datos}.ventas_diarias_estudio to usuario1")

In [ ]:
# Consultas adicionales por problema de permisos del usuario usuario1 en la base de datos data

In [ ]:
cursor.execute(f"drop table if exists classicmodels.ventas_diarias_estudio;")

In [ ]:
cursor.execute(f"CREATE TABLE classicmodels.ventas_diarias_estudio as SELECT * FROM {nombre_base_datos}.ventas_diarias_estudio")